# 束搜索
:label:`sec_beam-search`

在序列到序列學習中，我們逐個預測輸出序列，直到預測序列中出現特定的序列結束詞元"<eos>"。
本節將首先介紹*貪心搜索*（greedy search）策略，並探討其存在的問題，
然後對比其他替代策略：*窮舉搜索*（exhaustive search）和*束搜索*（beam search）。

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from d2l import torch as d2l

## 貪心搜索

對於輸出序列的每一時間步$t'$，我們都將基於貪心搜索找到具有最高條件概率的詞元：

$$y_{t'} = \operatorname*{argmax}_{y \in \mathcal{Y}} P(y \mid y_1, \ldots, y_{t'-1}, \mathbf{c})$$

一旦輸出序列包含了"<eos>"或者達到其最大長度$T'$，則輸出完成。

### 貪心搜索的問題

貪心搜索無法保證得到最優序列。最優序列應該是最大化
$\prod_{t'=1}^{T'} P(y_{t'} \mid y_1, \ldots, y_{t'-1}, \mathbf{c})$
值的輸出序列。

In [ ]:
def greedy_search(net, src_sentence, src_vocab, tgt_vocab, num_steps, device):
    """貪心搜索實現"""
    # 預處理源句子
    src_tokens = src_vocab[src_sentence.lower().split(' ')] + [src_vocab['<eos>']]
    enc_valid_len = torch.tensor([len(src_tokens)], device=device)
    src_tokens = d2l.truncate_pad(src_tokens, num_steps, src_vocab['<pad>'])
    
    # 編碼
    enc_X = torch.unsqueeze(torch.tensor(src_tokens, dtype=torch.long, device=device), dim=0)
    enc_outputs = net.encoder(enc_X, enc_valid_len)
    dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
    
    # 解碼
    dec_X = torch.unsqueeze(torch.tensor([tgt_vocab['<bos>']], dtype=torch.long, device=device), dim=0)
    output_seq, scores = [], []
    
    for _ in range(num_steps):
        Y, dec_state = net.decoder(dec_X, dec_state)
        # 選擇最高概率的詞元
        dec_X = Y.argmax(dim=2)
        pred = dec_X.squeeze(dim=0).type(torch.int32).item()
        
        # 記錄分數
        prob = Y.max(dim=2).values.item()
        scores.append(prob)
        
        if pred == tgt_vocab['<eos>']:
            break
        output_seq.append(pred)
    
    return output_seq, scores

## 束搜索實現

束搜索是貪心搜索的改進版本。它有一個超參數，名為*束寬*（beam size）$k$。
在時間步$1$，我們選擇具有最高條件概率的$k$個詞元。

In [ ]:
def beam_search(net, src_sentence, src_vocab, tgt_vocab, num_steps, device, beam_size=3):
    """束搜索實現
    
    參數:
        beam_size: 束寬，保留的候選序列數量
    
    返回:
        最佳序列和其分數
    """
    # 預處理源句子
    src_tokens = src_vocab[src_sentence.lower().split(' ')] + [src_vocab['<eos>']]
    enc_valid_len = torch.tensor([len(src_tokens)], device=device)
    src_tokens = d2l.truncate_pad(src_tokens, num_steps, src_vocab['<pad>'])
    
    # 編碼
    enc_X = torch.unsqueeze(torch.tensor(src_tokens, dtype=torch.long, device=device), dim=0)
    enc_outputs = net.encoder(enc_X, enc_valid_len)
    dec_state = net.decoder.init_state(enc_outputs, enc_valid_len)
    
    # 初始化束
    # 每個束包含: (序列, 累積對數概率, 解碼器狀態)
    beams = [([tgt_vocab['<bos>']], 0.0, dec_state)]
    completed = []  # 完成的序列
    
    for step in range(num_steps):
        candidates = []
        
        for seq, score, state in beams:
            # 如果序列已結束，保存並繼續
            if seq[-1] == tgt_vocab['<eos>']:
                completed.append((seq, score))
                continue
            
            # 解碼下一個詞元
            dec_X = torch.tensor([[seq[-1]]], dtype=torch.long, device=device)
            Y, new_state = net.decoder(dec_X, state)
            
            # 獲取top-k個詞元
            log_probs = torch.log_softmax(Y[0, -1], dim=0)
            topk_probs, topk_indices = torch.topk(log_probs, beam_size)
            
            # 為每個top-k詞元創建新的候選
            for i in range(beam_size):
                new_token = topk_indices[i].item()
                new_score = score + topk_probs[i].item()
                new_seq = seq + [new_token]
                candidates.append((new_seq, new_score, new_state))
        
        # 如果沒有候選了，退出
        if not candidates:
            break
        
        # 選擇分數最高的beam_size個候選
        beams = sorted(candidates, key=lambda x: x[1] / len(x[0]), reverse=True)[:beam_size]
    
    # 將未完成的束也加入completed
    completed.extend([(seq, score) for seq, score, _ in beams])
    
    # 返回分數最高的序列（考慮長度懲罰）
    alpha = 0.75  # 長度懲罰係數
    best_seq, best_score = max(completed, key=lambda x: x[1] / (len(x[0]) ** alpha))
    
    return best_seq[1:-1] if best_seq[-1] == tgt_vocab['<eos>'] else best_seq[1:], best_score

print("束搜索實現完成！")

## 可視化比較

讓我們可視化貪心搜索和束搜索的過程。

In [ ]:
def visualize_search_comparison(beam_sizes=[1, 2, 3, 5]):
    """可視化不同束寬的搜索結果"""
    
    # 創建一個簡單的示例來展示概念
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()
    
    for idx, beam_size in enumerate(beam_sizes):
        ax = axes[idx]
        
        # 模擬搜索過程
        steps = 5
        paths = beam_size
        
        # 生成隨機路徑分數
        np.random.seed(42)
        scores = np.random.rand(steps, paths)
        
        # 繪製路徑
        for p in range(paths):
            cumsum = np.cumsum(scores[:, p])
            ax.plot(range(steps), cumsum, marker='o', label=f'路徑 {p+1}')
        
        ax.set_title(f'束寬 = {beam_size}' if beam_size > 1 else '貪心搜索 (束寬=1)')
        ax.set_xlabel('時間步')
        ax.set_ylabel('累積分數')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('beam_search_comparison.png', dpi=150, bbox_inches='tight')
    print("可視化已保存為 beam_search_comparison.png")
    plt.show()

# 運行可視化
visualize_search_comparison()

## 🤖 AI 輔助學習指南

### 💡 核心概念理解

**Q: 束搜索和貪心搜索的主要區別是什麼？**

A: 
- **貪心搜索**：每步只保留1個最佳候選（beam_size=1）
- **束搜索**：每步保留k個最佳候選（beam_size=k）
- **優勢**：束搜索能探索更多可能性，找到更好的整體解決方案
- **代價**：計算量是貪心搜索的k倍

**Q: 如何選擇合適的束寬？**

A: 束寬選擇需要權衡：
- **太小（1-2）**：接近貪心搜索，可能錯過好的序列
- **適中（3-10）**：大多數任務的最佳選擇
- **太大（>20）**：計算成本高，收益遞減

**常見任務的推薦束寬：**
- 機器翻譯：4-10
- 圖像描述：3-5
- 對話生成：1-3（需要多樣性）

### 🔧 實現技巧

1. **長度歸一化**：使用 $\frac{1}{L^\alpha} \log P(y_1, \ldots, y_L \mid \mathbf{c})$，其中α=0.6-0.75
2. **早停策略**：當最佳候選遠超其他時，可以提前停止
3. **批量處理**：將k個候選作為一個批次並行處理，加速計算

### 📊 性能分析

| 方法 | 計算複雜度 | 結果質量 | 適用場景 |
|------|-----------|---------|----------|
| 貪心搜索 | O(T'·V) | 低 | 快速原型、實時應用 |
| 束搜索 (k=5) | O(5·T'·V) | 中-高 | 大多數生成任務 |
| 窮舉搜索 | O(V^T') | 最高 | 不可行（太慢） |

### 🎯 常見問題

**Q: 為什麼束搜索有時會生成重複的詞或短語？**

A: 解決方案：
- 添加**重複懲罰**：降低已生成詞元的概率
- 使用**n-gram阻止**：防止重複的n-gram
- 調整**溫度參數**：增加隨機性

**Q: 束搜索為什麼需要長度歸一化？**

A: 因為較短的序列往往有更高的聯合概率（概率連乘會變小），
長度歸一化確保不會總是偏向短序列。

### 💻 代碼優化建議

```python
# 優化1：批量處理
# 將k個候選序列組成一個批次，並行解碼
dec_X = torch.tensor([[seq[-1] for seq, _, _ in beams]], device=device)
Y, new_states = net.decoder(dec_X, states)

# 優化2：使用優先隊列
import heapq
beams = heapq.nlargest(beam_size, candidates, key=lambda x: x[1])

# 優化3：早停
if len(completed) >= beam_size and \
   best_score > max(score for _, score, _ in beams):
    break
```

### 📈 進階技巧

1. **多樣性束搜索**：鼓勵不同的候選序列，避免同質化
2. **約束束搜索**：強制包含特定詞元（如關鍵詞）
3. **分組束搜索**：將束分成多個組，組內競爭，增加多樣性

## 小結

* 序列搜索策略包括貪心搜索、窮舉搜索和束搜索。
* 貪心搜索所選取序列的計算量最小，但精度相對較低。
* 窮舉搜索所選取序列的精度最高，但計算量最大。
* 束搜索通過靈活選擇束寬，在正確率和計算代價之間進行權衡。
* 束寬的選擇需要根據具體任務和計算資源來決定。
* 長度歸一化對於生成高質量序列至關重要。

## 練習

1. 我們可以把窮舉搜索看作一種特殊的束搜索嗎？為什麼？
2. 在機器翻譯問題中應用束搜索。束寬是如何影響預測的速度和結果的？
3. 在語言模型中，基於用戶提供的前綴生成文本時，使用了哪種搜索策略？可以改進嗎？
4. 實現一個支持長度歸一化的束搜索算法。
5. 嘗試實現多樣性束搜索，比較其與標準束搜索的差異。

## 練習提示

**練習1提示：** 窮舉搜索可以看作束寬 = 詞表大小 的束搜索。

**練習2提示：** 使用前面章節的seq2seq模型，測試beam_size=1,3,5,10的效果。

**練習3提示：** 通常使用採樣策略或top-k採樣，而非確定性搜索。

**練習4提示：** 在計算序列分數時除以 $L^\alpha$，其中L是序列長度。

**練習5提示：** 在每個時間步對不同組的候選施加不同的懲罰，鼓勵多樣性。